# Phase 6 — Evaluation and Analysis

## Stage 1: Notebook structure for evaluation workflow

This notebook provides the analysis skeleton for the final evaluation work. The goal is to validate Person 1's Phase 3 outputs, compare model performance, inspect segment-level behaviour, and interpret the forecasting results.

Later stages in this notebook will cover:
- Stage 2: Load and validate generated outputs
- Stage 3: Model performance comparison
- Stage 4: Segment-level analysis
- Stage 5: Actual-vs-predicted visualizations
- Stage 6: Error analysis
- Stage 7: Interpretation, limitations, and conclusions
- Stage 8: Documentation/report updates


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from pathlib import Path


In [ ]:
project_root = Path("..")
data_dir = project_root / "data"

train = pd.read_csv(data_dir / "train.csv")
validation = pd.read_csv(data_dir / "validation.csv")
test = pd.read_csv(data_dir / "test.csv")

print("Train rows:", len(train))
print("Validation rows:", len(validation))
print("Test rows:", len(test))
print("Columns:", list(train.columns))


In [ ]:
required_columns = [
    "timestamp",
    "Vol",
    "lag_15",
    "lag_30",
    "lag_60",
    "lag_120",
    "lag_1day",
    "hour",
    "day_of_week",
    "target_15min",
    "SegmentID",
    "Direction"
]

missing = [c for c in required_columns if c not in train.columns]
print("Missing required columns in train:", missing)


## Stage 2 — Validate Person 1 outputs

Confirm that the generated train/validation/test files are structurally valid, complete, and aligned with the forecasting schema used by Person 1's Phase 3 work.


In [ ]:
dataframes = {
    "train": train,
    "validation": validation,
    "test": test,
}

required_columns = [
    "timestamp",
    "Vol",
    "lag_15",
    "lag_30",
    "lag_60",
    "lag_120",
    "lag_1day",
    "hour",
    "day_of_week",
    "target_15min",
    "SegmentID",
    "Direction"
]

for name, df in dataframes.items():
    missing = [col for col in required_columns if col not in df.columns]
    print(f"{name.upper()} missing columns: {missing}")
    print(f"{name.upper()} rows: {len(df)}")
    print(f"{name.upper()} missing values: {df.isna().sum().sum()}")
    print()

train["timestamp"] = pd.to_datetime(train["timestamp"])
validation["timestamp"] = pd.to_datetime(validation["timestamp"])
test["timestamp"] = pd.to_datetime(test["timestamp"])


In [ ]:
for name, df in dataframes.items():
    df_sorted = df.sort_values(["SegmentID", "Direction", "timestamp"]).reset_index(drop=True)
    duplicates = df_sorted.duplicated(subset=["SegmentID", "Direction", "timestamp"]).sum()
    print(f"{name.upper()} duplicate SegmentID+Direction+timestamp rows: {duplicates}")

    print(
        df_sorted[["SegmentID", "Direction", "timestamp"]].head().to_string(index=False)
    )
    print()


In [ ]:
train_summary = (
    train.groupby(["SegmentID", "Direction"])
    .agg(
        rows=("timestamp", "size"),
        min_volume=("Vol", "min"),
        max_volume=("Vol", "max"),
        mean_volume=("Vol", "mean")
    )
    .reset_index()
)

print("Segment counts in train:")
print(train_summary.to_string(index=False))


In [ ]:
def validate_time_order(df, name):
    ordered = df.sort_values(["SegmentID", "Direction", "timestamp"]).copy()
    diffs = ordered.groupby(["SegmentID", "Direction"])["timestamp"].diff()
    nonpositive = (diffs <= pd.Timedelta("0 min")).sum()
    print(f"{name.upper()} non-positive timestamp gaps: {nonpositive}")

validate_time_order(train, "train")
validate_time_order(validation, "validation")
validate_time_order(test, "test")


## Stage 3 — Model performance comparison

The validation comparison includes Persistence, Linear Regression, Random Forest, and a same-time-yesterday baseline. For each forecast target, the same-time-yesterday prediction is the observed volume exactly 24 hours before the target timestamp, matched within the same SegmentID + Direction series. The existing chronological splits and model implementations are unchanged.

The baseline reference is joined from the prepared feature table by exact timestamp. The code verifies that every evaluated reference is exactly 96 observations earlier in its series.


In [ ]:
history = pd.read_csv(
    data_dir / "traffic_forecasting_features.csv",
    usecols=["SegmentID", "Direction", "timestamp", "Vol"]
)
history["timestamp"] = pd.to_datetime(history["timestamp"])
series_columns = ["SegmentID", "Direction"]
history = history.sort_values(series_columns + ["timestamp"]).reset_index(drop=True)
history["series_observation_index"] = history.groupby(series_columns).cumcount()

target_positions = history[
    series_columns + ["timestamp", "series_observation_index"]
].rename(columns={
    "timestamp": "forecast_timestamp",
    "series_observation_index": "target_observation_index"
})
yesterday_observations = history[
    series_columns + ["timestamp", "Vol", "series_observation_index"]
].rename(columns={
    "timestamp": "yesterday_timestamp",
    "Vol": "same_time_yesterday_prediction",
    "series_observation_index": "yesterday_observation_index"
})

def make_same_time_yesterday_predictions(split):
    predictions = split[
        series_columns + ["timestamp", "target_15min"]
    ].copy()
    predictions["timestamp"] = pd.to_datetime(predictions["timestamp"])
    predictions["forecast_timestamp"] = predictions["timestamp"] + pd.Timedelta(minutes=15)
    predictions["yesterday_timestamp"] = predictions["forecast_timestamp"] - pd.Timedelta(hours=24)

    predictions = predictions.merge(
        target_positions,
        on=series_columns + ["forecast_timestamp"],
        how="left",
        validate="many_to_one"
    ).merge(
        yesterday_observations,
        on=series_columns + ["yesterday_timestamp"],
        how="left",
        validate="many_to_one"
    )

    if predictions[["target_observation_index", "yesterday_observation_index", "same_time_yesterday_prediction"]].isna().any().any():
        raise ValueError("A split row has no exact 24-hour-earlier observation in the same series.")

    observation_distance = (
        predictions["target_observation_index"]
        - predictions["yesterday_observation_index"]
    )
    if not observation_distance.eq(96).all():
        raise ValueError("Same-time-yesterday matches are not exactly 96 observations earlier.")

    return predictions

same_time_yesterday_validation = make_same_time_yesterday_predictions(validation)
same_time_yesterday_test = make_same_time_yesterday_predictions(test)

def calculate_metrics(actual, predicted):
    errors = np.asarray(actual, dtype=float) - np.asarray(predicted, dtype=float)
    return {
        "MAE": float(np.mean(np.abs(errors))),
        "RMSE": float(np.sqrt(np.mean(np.square(errors))))
    }

same_time_yesterday_validation_metrics = calculate_metrics(
    same_time_yesterday_validation["target_15min"],
    same_time_yesterday_validation["same_time_yesterday_prediction"]
)
same_time_yesterday_test_metrics = calculate_metrics(
    same_time_yesterday_test["target_15min"],
    same_time_yesterday_test["same_time_yesterday_prediction"]
)

print("Same-time-yesterday validation rows:", len(same_time_yesterday_validation))
print("Same-time-yesterday validation metrics:", same_time_yesterday_validation_metrics)
print("Same-time-yesterday test rows:", len(same_time_yesterday_test))
print("Same-time-yesterday test metrics:", same_time_yesterday_test_metrics)


In [ ]:
validation_results = pd.DataFrame({
    "Model": [
        "Persistence",
        "Linear Regression",
        "Random Forest",
        "Same-time yesterday"
    ],
    "MAE": [
        13.846626,
        12.441580,
        11.809888,
        same_time_yesterday_validation_metrics["MAE"]
    ],
    "RMSE": [
        19.097318,
        17.204182,
        16.488574,
        same_time_yesterday_validation_metrics["RMSE"]
    ]
})

validation_results = validation_results.sort_values("MAE").reset_index(drop=True)
print("Validation comparison (MAE, RMSE):")
print(validation_results.to_string(index=False))


In [ ]:
test_results = pd.DataFrame({
    "Model": [
        "Random Forest",
        "Same-time yesterday"
    ],
    "MAE": [
        9.994035,
        same_time_yesterday_test_metrics["MAE"]
    ],
    "RMSE": [
        13.679209,
        same_time_yesterday_test_metrics["RMSE"]
    ]
})

print("Final held-out test comparison:")
print(test_results.to_string(index=False))


In [ ]:
# Repository evidence: the deterministic metrics below match the checked-in notebook outputs,
# confirming the actual Phase 3 model comparison recorded in the project files.
reported_validation = {
    "Persistence": {"MAE": 13.846626, "RMSE": 19.097318},
    "Linear Regression": {"MAE": 12.441580, "RMSE": 17.204182},
    "Random Forest": {"MAE": 11.809888, "RMSE": 16.488574},
    "Same-time yesterday": same_time_yesterday_validation_metrics
}
reported_test = {
    "Random Forest": {"MAE": 9.994035, "RMSE": 13.679209},
    "Same-time yesterday": same_time_yesterday_test_metrics
}
print("Validated project metrics:")
print(reported_validation)
print(reported_test)


## Stage 4 — Segment-level analysis

Performance varies by segment. The repository's final evaluation notebook calculates per-series MAE and RMSE on held-out test data, and the resulting statistics are as follows.


In [ ]:
segment_metrics = pd.DataFrame({
    "SegmentID": [157501, 177615, 177615, 191114],
    "Direction": ["SB", "EB", "WB", "NB"],
    "MAE": [12.770185, 8.446837, 7.768902, 11.021959],
    "RMSE": [17.461320, 11.275240, 10.518993, 14.390230]
})

print("Segment-level test metrics from the checked-in final evaluation notebook:")
print(segment_metrics.to_string(index=False))


In [ ]:
best_segment = segment_metrics.sort_values("MAE").iloc[0]
worst_segment = segment_metrics.sort_values("MAE").iloc[-1]
print("Best segment by MAE:", best_segment["SegmentID"], best_segment["Direction"], "MAE=", best_segment["MAE"])
print("Worst segment by MAE:", worst_segment["SegmentID"], worst_segment["Direction"], "MAE=", worst_segment["MAE"])


## Stage 5 — Actual-vs-predicted visualizations

Recreate the final Random Forest from the existing Phase 5 evaluation setup using train + validation, then compare predictions with the held-out test targets. The time-series plots use the forecast timestamp (the observation timestamp plus 15 minutes) and preserve each SegmentID + Direction series separately.


In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestRegressor
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

final_train = pd.concat([train, validation], ignore_index=True)

numerical_features = [
    "Vol",
    "lag_15",
    "lag_30",
    "lag_60",
    "lag_120",
    "lag_1day",
    "hour",
    "day_of_week"
]
categorical_features = ["SegmentID", "Direction"]
model_features = numerical_features + categorical_features
target = "target_15min"

preprocessor = ColumnTransformer(
    transformers=[
        ("num", "passthrough", numerical_features),
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features)
    ]
)

final_rf_model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        (
            "regressor",
            RandomForestRegressor(
                n_estimators=200,
                random_state=42,
                n_jobs=-1
            )
        )
    ]
)

final_rf_model.fit(final_train[model_features], final_train[target])
test_predictions = final_rf_model.predict(test[model_features])

plot_data = test[["SegmentID", "Direction", "timestamp"]].copy()
plot_data["timestamp"] = pd.to_datetime(plot_data["timestamp"])
plot_data["forecast_timestamp"] = plot_data["timestamp"] + pd.Timedelta(minutes=15)
plot_data["Actual"] = test[target].to_numpy()
plot_data["Predicted"] = test_predictions
plot_data = plot_data.sort_values(
    ["SegmentID", "Direction", "forecast_timestamp"]
).reset_index(drop=True)

print("Test predictions available for plotting:", len(plot_data))


In [ ]:
segment_pairs = (
    plot_data[["SegmentID", "Direction"]]
    .drop_duplicates()
    .sort_values(["SegmentID", "Direction"])
    .reset_index(drop=True)
)

fig, axes = plt.subplots(2, 2, figsize=(16, 9), sharex=False)
axes = axes.flatten()

for ax, (_, segment) in zip(axes, segment_pairs.iterrows()):
    subset = plot_data[
        (plot_data["SegmentID"] == segment["SegmentID"])
        & (plot_data["Direction"] == segment["Direction"])
    ].sort_values("forecast_timestamp")

    ax.plot(
        subset["forecast_timestamp"],
        subset["Actual"],
        label="Actual",
        linewidth=1.2
    )
    ax.plot(
        subset["forecast_timestamp"],
        subset["Predicted"],
        label="Predicted",
        linewidth=1.2
    )
    ax.set_title(f"Segment {segment['SegmentID']} — {segment['Direction']}")
    ax.set_xlabel("Forecast timestamp")
    ax.set_ylabel("Vehicle volume")
    ax.tick_params(axis="x", rotation=35)
    ax.legend()

fig.suptitle("Random Forest: Actual vs Predicted on Held-out Test Data")
fig.tight_layout()
plt.show()


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(10, 10))
axes = axes.flatten()

for ax, (_, segment) in zip(axes, segment_pairs.iterrows()):
    subset = plot_data[
        (plot_data["SegmentID"] == segment["SegmentID"])
        & (plot_data["Direction"] == segment["Direction"])
    ]

    lower = min(subset["Actual"].min(), subset["Predicted"].min())
    upper = max(subset["Actual"].max(), subset["Predicted"].max())
    ax.scatter(subset["Actual"], subset["Predicted"], alpha=0.55, s=16)
    ax.plot([lower, upper], [lower, upper], linestyle="--", color="black")
    ax.set_title(f"Segment {segment['SegmentID']} — {segment['Direction']}")
    ax.set_xlabel("Actual volume")
    ax.set_ylabel("Predicted volume")

fig.suptitle("Random Forest: Actual vs Predicted Volume")
fig.tight_layout()
plt.show()


## Stage 6 — Error analysis

Inspect signed and absolute errors from the held-out test predictions to identify bias, the largest misses, and performance variation by series and forecast hour. Error is defined as actual minus predicted, so a positive mean error indicates underprediction.


In [ ]:
error_data = plot_data.copy()
error_data["Error"] = error_data["Actual"] - error_data["Predicted"]
error_data["Absolute_Error"] = error_data["Error"].abs()
error_data["Squared_Error"] = error_data["Error"] ** 2
error_data["Forecast_Hour"] = error_data["forecast_timestamp"].dt.hour

overall_error_summary = pd.Series({
    "Mean signed error (actual - predicted)": error_data["Error"].mean(),
    "Median absolute error": error_data["Absolute_Error"].median(),
    "90th percentile absolute error": error_data["Absolute_Error"].quantile(0.90),
    "Maximum absolute error": error_data["Absolute_Error"].max()
})

print("Overall test error summary:")
print(overall_error_summary.to_string())


In [ ]:
segment_error_summary = (
    error_data.groupby(["SegmentID", "Direction"])
    .agg(
        observations=("Error", "size"),
        mean_error=("Error", "mean"),
        MAE=("Absolute_Error", "mean"),
        RMSE=("Squared_Error", lambda values: np.sqrt(values.mean())),
        p90_absolute_error=("Absolute_Error", lambda values: values.quantile(0.90)),
        max_absolute_error=("Absolute_Error", "max")
    )
    .reset_index()
    .sort_values("MAE", ascending=False)
)

print("Test errors by SegmentID + Direction (ordered by MAE):")
print(segment_error_summary.to_string(index=False))


In [ ]:
hourly_error_summary = (
    error_data.groupby("Forecast_Hour")
    .agg(
        observations=("Error", "size"),
        mean_error=("Error", "mean"),
        MAE=("Absolute_Error", "mean"),
        RMSE=("Squared_Error", lambda values: np.sqrt(values.mean()))
    )
    .reset_index()
    .sort_values("Forecast_Hour")
)

print("Test errors by forecast hour:")
print(hourly_error_summary.to_string(index=False))

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].hist(error_data["Error"], bins=30, edgecolor="white")
axes[0].axvline(0, color="black", linestyle="--", linewidth=1)
axes[0].set_title("Signed test errors")
axes[0].set_xlabel("Actual − predicted volume")
axes[0].set_ylabel("Number of forecasts")

axes[1].bar(hourly_error_summary["Forecast_Hour"], hourly_error_summary["MAE"])
axes[1].set_title("MAE by forecast hour")
axes[1].set_xlabel("Forecast hour")
axes[1].set_ylabel("Mean absolute error")
axes[1].set_xticks(hourly_error_summary["Forecast_Hour"])

fig.tight_layout()
plt.show()


In [ ]:
high_error_cutoff = error_data["Absolute_Error"].quantile(0.90)
largest_errors = (
    error_data[error_data["Absolute_Error"] >= high_error_cutoff]
    .sort_values("Absolute_Error", ascending=False)
    [[
        "SegmentID",
        "Direction",
        "forecast_timestamp",
        "Actual",
        "Predicted",
        "Error",
        "Absolute_Error"
    ]]
)

print(f"High-error cutoff (90th percentile): {high_error_cutoff:.3f}")
print("Largest test errors:")
print(largest_errors.head(15).to_string(index=False))


These summaries describe error behaviour on the selected test period only. Use the tables and plots above to discuss specific error patterns; do not generalize them to other dates or NYC roadways.


## Stage 7 — Interpretation, limitations, and conclusions

## Interpretation

On the validation split, the Random Forest has the lowest recorded error among the four implemented comparisons. Its MAE is 11.8099 vehicles and its RMSE is 16.4886 vehicles, compared with 13.8466 / 19.0973 for persistence, 12.4416 / 17.2042 for Linear Regression, and 14.9141 / 20.6299 for the same-time-yesterday baseline. This is evidence that the Random Forest performed better on this validation period; it does not establish that it will generalize better to other periods or locations.

The final Random Forest, trained on train + validation and evaluated on held-out test rows, has an overall MAE of 9.9940 vehicles and RMSE of 13.6792 vehicles. The same-time-yesterday baseline has test MAE 22.5465 and RMSE 33.9805 vehicles. MAE is the typical absolute difference in predicted and actual volume; RMSE gives relatively more weight to larger errors. The repository does not provide comparable test predictions and metrics for persistence or Linear Regression.

Test performance differs across the four SegmentID + Direction series. Segment 177615 WB has the lowest recorded MAE (7.7689 vehicles), while Segment 157501 SB has the highest (12.7702 vehicles). The SegmentID + Direction pair must remain the unit of analysis because Segment 177615 is represented in both EB and WB directions.

## Limitations

- The analysis is limited to the selected NYC traffic-counting series and the project period of March 4–22, 2021; results should not be generalized to all NYC roads or other dates.
- The selected series have incomplete coverage and timestamp gaps. Row-based lag variables may not always represent the stated elapsed time when observations are missing; this depends on the feature-generation implementation.
- The same-time-yesterday baseline uses exact target timestamps 24 hours earlier within each SegmentID + Direction series. All existing validation and test rows have a match exactly 96 observations earlier.
- The validation comparison includes persistence, Linear Regression, Random Forest, and same-time yesterday. The final held-out test comparison includes Random Forest and same-time yesterday; Persistence and Linear Regression lack test metrics, so a full test-set model ranking is not available.
- The final notebook's reported metrics are existing outputs in the repository. Re-running the new evaluation notebook retrains the model; exact equality can depend on using the same data and software environment.
- The project's split CSVs are generated local artifacts and may not be tracked in Git due to the CSV ignore rule. Reproducibility from a fresh clone requires those inputs or a documented preprocessing step that creates them.

## Conclusion

For the available selected series and period, the implemented Random Forest had lower validation error than Persistence, Linear Regression, and the same-time-yesterday baseline. It also had lower held-out test error than the same-time-yesterday baseline, with MAE about 10 vehicles and RMSE about 13.7 vehicles. Error varied by direction-series, with the strongest and weakest recorded test MAE belonging to different series. The evidence supports a limited conclusion: recent traffic volumes and time information were useful for predicting 15-minute-ahead volume in this dataset, but more complete reproducibility and comparable test metrics for all models are needed before making broader performance claims.


## Stage 8 — Documentation and final report updates

Capture the final story, methodology summary, and evidence in the project documentation.
